# ViSoBERT trên ViGoEmotions — 7 nhóm cảm xúc (Ekman + Neutral)



## 1. Cài đặt thư viện

In [ ]:
!pip install -q transformers==4.52.4 sentencepiece

## 2. Cấu hình và đường dẫn

In [ ]:

import os, sys, json, re, time, random
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# ======================= CẤU HÌNH =======================
SEED        = 42
EPOCHS      = 12
BATCH_SIZE  = 32
LR          = 5e-5
MAX_LEN     = 200
THRESHOLD   = 0.5

MODEL_NAME  = 'uitnlp/visobert'
MODEL_TAG   = 'visobert'

USE_PYVI    = True
POOLING     = 'pooler'
USE_AMP     = False

EARLY_STOP_PATIENCE = 3
EARLY_STOP_MIN_DELTA = 0.0
RESUME      = True
DELETE_LAST_WHEN_DONE = True
SMOKE_TEST  = False
RUN_TAG     = ''

DATA_TAG     = 'data_ekman7'
BASELINE_RUN = 'visobert_seed42_pooler_4.52Transformer'

# =========================================================

if SMOKE_TEST:
    EPOCHS = 1

RUN_NAME = f"{MODEL_TAG}_s{SEED}_{POOLING}"
# ======================= REPO PATH =======================

COLAB_REPO_ROOT = '/content/drive/MyDrive/DAP391m-sentiment-vn'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path(COLAB_REPO_ROOT)
else:
    REPO_ROOT = Path('..')

    if not (REPO_ROOT / 'notebooks').is_dir():
        raise RuntimeError(
            f'Không thấy {REPO_ROOT.resolve()}/notebooks. '
            'Hãy chạy notebook từ thư mục notebooks/ của repo.'
        )

DATA_ROOT = (REPO_ROOT / 'data' / 'processed').as_posix()
OUT_ROOT  = (REPO_ROOT / 'runs').as_posix()

DATA_DIR = f'{DATA_ROOT}/{DATA_TAG}'
OUT_DIR  = f'{OUT_ROOT}/{RUN_NAME}'

os.makedirs(OUT_DIR, exist_ok=True)

for f in [
    'train.csv',
    'val.csv',
    'test.csv',
    'label_dict_ekman7.json',
    'ekman_mapping.json'
]:
    assert os.path.exists(f'{DATA_DIR}/{f}'), (
        f'Thiếu {DATA_DIR}/{f} — hãy chạy notebook tiền xử lý trước.'
    )

CONFIG = dict(
    seed=SEED,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_len=MAX_LEN,
    threshold=THRESHOLD,
    model_name=MODEL_NAME,
    model_tag=MODEL_TAG,
    use_pyvi=USE_PYVI,
    pooling=POOLING,
    use_amp=USE_AMP,
    smoke=SMOKE_TEST,
    label_scheme='ekman7',
    data_tag=DATA_TAG,
    early_stop_patience=EARLY_STOP_PATIENCE,
    early_stop_min_delta=EARLY_STOP_MIN_DELTA
)

with open(f'{OUT_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print('Repo root:', REPO_ROOT.resolve())
print('Run:', RUN_NAME)
print('Dữ liệu:', DATA_DIR)
print('Output dir:', OUT_DIR)


## 3. Import, GPU, seed, ghi lại môi trường

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
import transformers
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'KHÔNG CÓ GPU - hãy bật GPU trong Runtime')

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

env = dict(python=sys.version.split()[0], torch=torch.__version__, transformers=transformers.__version__,
           pandas=pd.__version__, numpy=np.__version__,
           gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else None)
import sklearn; env['scikit-learn'] = sklearn.__version__
with open(f'{OUT_DIR}/env.json', 'w') as f:
    json.dump(env, f, indent=2)
print(env)

## 4. Đọc dữ liệu đã tiền xử lý (7 nhóm)

In [ ]:
with open(f'{DATA_DIR}/label_dict_ekman7.json', 'r') as f:
    label_dict = json.load(f)               # {"0": "joy", ..., "6": "neutral"}
with open(f'{DATA_DIR}/ekman_mapping.json', 'r') as f:
    mapping = json.load(f)

N_LABELS = len(label_dict)
LABEL_NAMES = [label_dict[str(i)] for i in range(N_LABELS)]
assert LABEL_NAMES == mapping['group_names']
print('Số nhãn:', N_LABELS, LABEL_NAMES)

def read_split(name):
    return pd.read_csv(f'{DATA_DIR}/{name}.csv', dtype={'id': str}, keep_default_na=False)

train_df, val_df, test_df = read_split('train'), read_split('val'), read_split('test')

if SMOKE_TEST:
    train_df, val_df, test_df = train_df.head(256), val_df.head(64), test_df.head(64)

TEXT_COL = 'model_text' if USE_PYVI else 'text'
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    assert TEXT_COL in d.columns, f'{name} thiếu cột {TEXT_COL} (notebook tiền xử lý chạy với USE_PYVI=False?)'
    assert (d[TEXT_COL].str.len() > 0).all(), f'{name} có văn bản rỗng'
    print(name, d.shape)
train_df[[TEXT_COL, 'labels', 'label_names']].head(3)

## 5. Mã hóa nhãn (multi-hot, 7 nhóm)

In [ ]:
def encode_labels(s):
    ids = [int(x) for x in re.findall(r'\d+', str(s))]
    assert ids and max(ids) < N_LABELS, s
    v = np.zeros(N_LABELS, dtype=np.float32)
    v[ids] = 1.0
    return v

Y_train = np.stack(train_df['labels'].apply(encode_labels).values)
Y_val   = np.stack(val_df['labels'].apply(encode_labels).values)
Y_test  = np.stack(test_df['labels'].apply(encode_labels).values)

label_counts = Y_train.sum(axis=0)
print('Tần suất nhóm (train):', dict(zip(LABEL_NAMES, label_counts.astype(int))))
print(f'Tỉ lệ lớn nhất/nhỏ nhất: {label_counts.max()/max(label_counts.min(),1):.1f}x | số nhóm/mẫu trung bình: {Y_train.sum(1).mean():.2f} | '
      f'mẫu có >=2 nhóm: {(Y_train.sum(1) >= 2).mean()*100:.1f}%')
plt.figure(figsize=(7, 3.5)); plt.bar(LABEL_NAMES, label_counts); plt.xticks(rotation=45); plt.title('Phân bố nhóm (train)'); plt.tight_layout(); plt.show()

## 6. Tokenizer và DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def encode(texts):
    enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN, padding='max_length',
                    return_attention_mask=True, return_token_type_ids=False, return_tensors='pt')
    return enc['input_ids'], enc['attention_mask']

t0 = time.time()
tr_ids, tr_mask = encode(train_df[TEXT_COL])
va_ids, va_mask = encode(val_df[TEXT_COL])
te_ids, te_mask = encode(test_df[TEXT_COL])
print(f'Tokenize xong ({time.time()-t0:.1f}s)')

# Bản gốc chọn max_len=200 dựa trên số KÝ TỰ, nhưng dùng như số TOKEN. Kiểm tra số token thực tế:
lens = tr_mask.sum(dim=1).numpy()
print(f'Token/mẫu (train): trung bình {lens.mean():.1f} | p95 {np.percentile(lens,95):.0f} | p99 {np.percentile(lens,99):.0f} | '
      f'bị cắt (chạm {MAX_LEN}): {(lens >= MAX_LEN).mean()*100:.2f}%')

train_ds = TensorDataset(tr_ids, tr_mask, torch.from_numpy(Y_train))
val_ds   = TensorDataset(va_ids, va_mask, torch.from_numpy(Y_val))
test_ds  = TensorDataset(te_ids, te_mask, torch.from_numpy(Y_test))

train_gen = torch.Generator(); train_gen.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  generator=train_gen, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print('Số batch/epoch:', len(train_loader))

## 7. Mô hình

In [ ]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_name, pooling='pooler'):
        super().__init__()
        self.pooling = pooling
        config = AutoConfig.from_pretrained(model_name, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1)
        self.bert = AutoModel.from_pretrained(model_name, config=config)   # cảnh báo "pooler newly initialized" là bình thường (và là điểm đáng chú ý)
        self.drop = nn.Dropout(p=0.2)
        self.fc = nn.Linear(self.bert.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        x = out.pooler_output if self.pooling == 'pooler' else out.last_hidden_state[:, 0]
        return self.fc(self.drop(x))

set_seed(SEED)   # khởi tạo pooler + fc phụ thuộc seed
model = ModelSentimentClassifier(N_LABELS, MODEL_NAME, POOLING).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Tổng tham số: {n_params:,}')

## 8. Optimizer, scheduler, loss

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR)
steps_per_epoch = len(train_loader)
scheduler = get_scheduler('linear', optimizer=optimizer, num_warmup_steps=steps_per_epoch,
                          num_training_steps=steps_per_epoch * EPOCHS)

# pos_weight tính từ train (clip count>=1 chỉ để smoke test không chia cho 0; không ảnh hưởng dữ liệu đầy đủ)
cnt = np.clip(label_counts, 1, None)
pos_weight = torch.tensor((len(Y_train) - cnt) / cnt, dtype=torch.float32).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and device.type == 'cuda'))

## 9. Hàm train / eval

In [ ]:
def run_epoch(loader, train=False, return_probs=False):
    model.train(train)
    losses, all_t, all_p = [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for ids, mask, y in loader:
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(USE_AMP and device.type == 'cuda')):
                logits = model(ids, mask)
            loss = loss_fn(logits.float(), y)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer); scaler.update()
                scheduler.step()
            losses.append(loss.item())
            all_t.append(y.cpu().numpy()); all_p.append(torch.sigmoid(logits.float()).detach().cpu().numpy())
    T, P = np.vstack(all_t), np.vstack(all_p)
    pred = (P >= THRESHOLD).astype(int)
    acc = (T == pred).mean(axis=0).mean()
    f1 = f1_score(T, pred, average='macro', zero_division=0)
    out = (float(np.mean(losses)), float(acc), float(f1))
    return out + (P, T) if return_probs else out

def save_atomic(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp); os.replace(tmp, path)

## 10. Huấn luyện (resume được, có early stopping)

Early stopping theo val Macro-F1: dừng khi không cải thiện `EARLY_STOP_PATIENCE` epoch liên tiếp (mặc định 3). `best.pt` luôn là checkpoint tốt nhất theo val, và mục 12 đánh giá từ `best.pt`, nên dừng sớm không làm lệch kết quả test. Lưu ý lịch giảm lr được tính cho `EPOCHS` epoch, nên khi dừng sớm lr vẫn chưa về 0. Nếu phiên Colab bị ngắt: mở lại, chạy lại từ đầu (Run all) — notebook sẽ tự tiếp tục từ `last.pt` với cùng `SEED`/cấu hình. Nếu đổi cấu hình mà không đổi `RUN_NAME`, notebook sẽ báo lỗi thay vì resume nhầm.

In [ ]:
LAST, BEST, HIST, DONE = (f'{OUT_DIR}/{n}' for n in ['last.pt', 'best.pt', 'history.json', 'DONE'])

history, best_f1, start_epoch = [], 0.0, 0
bad_epochs, stopped = 0, False   # early stopping

if os.path.exists(DONE) and RESUME:
    print('Run này đã hoàn tất trước đó -> bỏ qua huấn luyện (xóa file DONE hoặc đổi SEED để chạy lại).')
    history = json.load(open(HIST))
    best_f1 = max(h['val_f1'] for h in history)
    start_epoch = EPOCHS
elif os.path.exists(LAST) and RESUME:
    ck = torch.load(LAST, map_location='cpu', weights_only=False)
    assert ck['config'] == CONFIG, f"Cấu hình checkpoint khác cấu hình hiện tại:\n{ck['config']}\n{CONFIG}"
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
    history, best_f1, start_epoch = ck['history'], ck['best_f1'], ck['epoch'] + 1
    bad_epochs = ck.get('bad_epochs', 0)
    stopped = EARLY_STOP_PATIENCE > 0 and bad_epochs >= EARLY_STOP_PATIENCE   # lần chạy trước đã early-stop đúng epoch cuối
    train_gen.set_state(ck['gen_state']); torch.set_rng_state(ck['rng_cpu'])
    if torch.cuda.is_available() and ck['rng_cuda'] is not None: torch.cuda.set_rng_state_all(ck['rng_cuda'])
    random.setstate(ck['rng_py']); np.random.set_state(ck['rng_np'])
    print(f'Resume từ epoch {start_epoch + 1}/{EPOCHS} (best val F1 hiện tại: {best_f1:.4f})')

t_start = time.time()
for epoch in range(start_epoch, EPOCHS):
    if stopped: break
    t0 = time.time()
    tr_loss, tr_acc, tr_f1 = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_f1 = run_epoch(val_loader, train=False)
    history.append(dict(epoch=epoch + 1, lr=optimizer.param_groups[0]['lr'], train_loss=tr_loss, train_acc=tr_acc, train_f1=tr_f1,
                        val_loss=va_loss, val_acc=va_acc, val_f1=va_f1))
    improved = va_f1 > best_f1 + EARLY_STOP_MIN_DELTA
    bad_epochs = 0 if improved else bad_epochs + 1
    if improved:
        best_f1 = va_f1
        save_atomic(dict(model=model.state_dict(), epoch=epoch + 1, val_f1=va_f1, config=CONFIG), BEST)
    save_atomic(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                     scaler=scaler.state_dict(), epoch=epoch, best_f1=best_f1, bad_epochs=bad_epochs, history=history, config=CONFIG,
                     gen_state=train_gen.get_state(), rng_cpu=torch.get_rng_state(),
                     rng_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                     rng_py=random.getstate(), rng_np=np.random.get_state()), LAST)
    json.dump(history, open(HIST, 'w'), indent=1)
    print(f'Epoch {epoch+1:>2}/{EPOCHS} | {time.time()-t0:5.0f}s | train loss {tr_loss:.4f} F1 {tr_f1:.4f} | '
          f'val loss {va_loss:.4f} F1 {va_f1:.4f}' + (' | *best*' if improved else f' | không cải thiện {bad_epochs}/{EARLY_STOP_PATIENCE}' if EARLY_STOP_PATIENCE > 0 else ''))
    if EARLY_STOP_PATIENCE > 0 and bad_epochs >= EARLY_STOP_PATIENCE:
        stopped = True
        print(f'Early stopping tại epoch {epoch+1}: val Macro-F1 không cải thiện {EARLY_STOP_PATIENCE} epoch liên tiếp (best {best_f1:.4f}).')
        break

if start_epoch < EPOCHS or stopped:
    open(DONE, 'w').write('done')
    if DELETE_LAST_WHEN_DONE and os.path.exists(LAST):
        os.remove(LAST)
    print(f'Hoàn tất. Thời gian phiên này: {(time.time()-t_start)/60:.1f} phút')

## 11. Đường cong huấn luyện

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['epoch'], h['train_loss'], label='train'); ax[0].plot(h['epoch'], h['val_loss'], label='val'); ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid()
ax[1].plot(h['epoch'], h['train_f1'], label='train'); ax[1].plot(h['epoch'], h['val_f1'], label='val'); ax[1].set_title('Macro-F1 @ ngưỡng 0.5'); ax[1].legend(); ax[1].grid()
plt.tight_layout(); plt.show()

## 12. Đánh giá trên test + lưu xác suất

In [ ]:
ck = torch.load(BEST, map_location='cpu', weights_only=False)
model.load_state_dict(ck['model']); model.to(device)
print(f"Checkpoint tốt nhất: epoch {ck['epoch']} (val Macro-F1 {ck['val_f1']:.4f})")

_, _, val_f1, P_val, T_val   = run_epoch(val_loader,  return_probs=True)
_, _, test_f1, P_test, T_test = run_epoch(test_loader, return_probs=True)

# Lưu xác suất để thử per-label threshold / calibration mà không cần train lại
np.save(f'{OUT_DIR}/val_probs.npy', P_val);   np.save(f'{OUT_DIR}/val_labels.npy', T_val)
np.save(f'{OUT_DIR}/test_probs.npy', P_test); np.save(f'{OUT_DIR}/test_labels.npy', T_test)

pred = (P_test >= THRESHOLD).astype(int)
rep = classification_report(T_test, pred, target_names=LABEL_NAMES, zero_division=0, output_dict=True)
pd.DataFrame(rep).transpose().to_excel(f'{OUT_DIR}/classification_report_test.xlsx')

print(classification_report(T_test, pred, target_names=LABEL_NAMES, zero_division=0))
macro, weighted = rep['macro avg']['f1-score'] * 100, rep['weighted avg']['f1-score'] * 100
micro   = f1_score(T_test, pred, average='micro',   zero_division=0) * 100
samples = f1_score(T_test, pred, average='samples', zero_division=0) * 100
exact   = (pred == T_test).all(axis=1).mean() * 100
print(f'TEST (7 nhóm)  Macro-F1 = {macro:.2f} | Weighted-F1 = {weighted:.2f} | Micro-F1 = {micro:.2f} | Samples-F1 = {samples:.2f} | Exact match = {exact:.2f}')
print('(Không so với 61.50 / 63.26 của paper: đó là 28 nhãn. Xem mục 14 để so sánh công bằng.)')
if SMOKE_TEST:
    print('(SMOKE_TEST: số liệu trên chỉ để kiểm tra pipeline, không có ý nghĩa.)')

## 13. Ma trận nhầm lẫn theo từng nhóm (multi-label)

Mỗi ô là tỉ lệ mẫu **thật sự có nhóm hàng** mà mô hình **dự đoán có nhóm cột** (đường chéo = recall). Giá trị ngoài đường chéo cao cho biết hai nhóm hay bị trộn lẫn; đó là chỗ để xem lại định nghĩa nhóm.

In [ ]:
T_i = T_test.astype(int)
M = np.zeros((N_LABELS, N_LABELS))
for i in range(N_LABELS):
    rows = T_i[:, i] == 1
    M[i] = pred[rows].mean(axis=0) if rows.any() else 0
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(M, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N_LABELS)); ax.set_xticklabels(LABEL_NAMES, rotation=45); ax.set_yticks(range(N_LABELS)); ax.set_yticklabels(LABEL_NAMES)
ax.set_xlabel('dự đoán có nhóm'); ax.set_ylabel('thật sự có nhóm')
for i in range(N_LABELS):
    for j in range(N_LABELS): ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', fontsize=8, color='white' if M[i,j] > .6 else 'black')
plt.colorbar(im); plt.tight_layout(); plt.show()

## 14. So sánh công bằng với run 28 nhãn (cùng tập test)

Cách làm: lấy xác suất test của run 28 nhãn đã tái lập, với mỗi nhóm lấy **max xác suất của các nhãn thành viên** rồi áp ngưỡng 0.5 (tương đương “dự đoán có nhóm nếu có ít nhất một nhãn thành viên được dự đoán”). Nhãn thật cũng được gộp theo cùng luật. Sau đó so với mô hình huấn luyện thẳng trên 7 nhóm.

Cell này kiểm tra nhãn thật gộp từ 28 nhãn phải **khớp từng dòng** với nhãn 7 nhóm ở trên (xác nhận thứ tự test không đổi). Nếu không thấy file của run 28 nhãn thì bỏ qua.

Cách đọc kết quả: nếu mô hình huấn luyện thẳng 7 nhóm không hơn rõ rệt phương án “train 28 nhãn rồi gộp”, thì việc gom nhãn chỉ giúp đơn giản hóa bài toán, chưa giúp mô hình học tốt hơn. Một seed chênh vài phần trăm F1 chưa đủ kết luận; cần ≥ 3 seed cho mỗi phía.

In [ ]:
BASE_DIR = f'{OUT_ROOT}/{BASELINE_RUN}'
if SMOKE_TEST:
    print('SMOKE_TEST: bỏ qua so sánh.')
elif not os.path.exists(f'{BASE_DIR}/test_probs.npy'):
    print(f'Không thấy {BASE_DIR}/test_probs.npy — bỏ qua. Chỉnh BASELINE_RUN ở mục 2 nếu run 28 nhãn có tên khác.')
else:
    P28 = np.load(f'{BASE_DIR}/test_probs.npy'); T28 = np.load(f'{BASE_DIR}/test_labels.npy')
    n_orig = len(mapping['orig_label_dict']); assert P28.shape[1] == n_orig
    A = np.zeros((n_orig, N_LABELS))
    for i, g in mapping['orig_id_to_group_id'].items(): A[int(i), g] = 1

    T_grp = ((T28 @ A) > 0).astype(int)
    assert T_grp.shape == T_test.shape and (T_grp == T_test.astype(int)).all(), 'Nhãn thật gộp từ run 28 nhãn không khớp tập test hiện tại'
    P_grp = np.stack([P28[:, A[:, g] > 0].max(axis=1) for g in range(N_LABELS)], axis=1)
    pred_grp = (P_grp >= THRESHOLD).astype(int)

    def summarize(pred_, name):
        r = classification_report(T_test, pred_, target_names=LABEL_NAMES, zero_division=0, output_dict=True)
        d = {k: r[k]['f1-score'] * 100 for k in LABEL_NAMES}
        d.update({'Macro-F1': r['macro avg']['f1-score'] * 100, 'Weighted-F1': r['weighted avg']['f1-score'] * 100,
                  'Micro-F1': f1_score(T_test, pred_, average='micro', zero_division=0) * 100,
                  'Samples-F1': f1_score(T_test, pred_, average='samples', zero_division=0) * 100})
        return pd.Series(d, name=name)

    cmp_df = pd.concat([summarize(pred_grp, '28 nhãn -> gộp 7'), summarize(pred, 'train thẳng 7 nhóm')], axis=1)
    cmp_df['chênh lệch'] = cmp_df.iloc[:, 1] - cmp_df.iloc[:, 0]
    cmp_df.round(2).to_excel(f'{OUT_DIR}/compare_vs_28label_test.xlsx')
    display(cmp_df.round(2))

## 15. Bước tiếp theo

- Chạy lại với `SEED = 1, 2, ...` (cả run 7 nhóm và run 28 nhãn) rồi báo mean ± std Macro-F1 trên test.
- Per-label threshold: dùng `val_probs.npy`/`val_labels.npy` để chọn ngưỡng từng nhóm, rồi áp lên `test_probs.npy` — không cần train lại. Với 7 nhóm mất cân bằng (Joy ≈ 49% mẫu, Neutral ≈ 5%), bước này nhiều khả năng đáng làm hơn so với bản 28 nhãn.
- Ablation rẻ: `USE_PYVI = False`, `POOLING = 'cls'`.
- Nếu muốn thử bỏ nhiễu Neutral đi kèm nhãn khác (~31% câu Neutral ở train còn mang nhóm khác), làm ở notebook tiền xử lý và chạy lại như một biến thể riêng, không sửa đè lên bản này.